# 실전 11-1: MCP (Model Context Protocol) 기본 아키텍처

## 1. MCP란 무엇인가?
- 2024년 말 Anthropic에서 오픈소스로 발표한 **AI 에이전트와 외부 데이터/도구(Tools) 간의 범용 표준 통신 규약**입니다.
- 과거에는 OpenAI, Anthropic, Google 등 모델마다 Tool Calling 방식이 다르고, 노션/깃허브/슬랙 등 데이터 소스마다 별도의 API 코드를 짜야 했습니다.
- **MCP의 혁신**: USB-C 타입 케이블처럼, 챗봇 클라이언트와 데이터 서버 사이에 '표준 케이블(MCP)'을 꽂기만 하면 어떤 모델이든 세상의 모든 데이터에 즉시 접근할 수 있게 됩니다.

## 2. MCP의 구조 (Client - Server)
MCP는 두 가지 핵심 요소로 이루어져 있습니다.
1. **MCP Host (Client)**: LLM이 실행되는 환경 (예: Claude Desktop 앱, LangChain/LangGraph 에이전트, Cursor IDE 등)
2. **MCP Server**: 실제 외부 데이터가 있는 곳 (예: 로컬 파일 시스템, Github, SQLite 데이터베이스 등)

이 노트북에서는 파이썬으로 가상의 **가장 단순한 MCP Server**를 만들고, 랭체인으로 구성한 **MCP Client**가 이 서버에 접근하여 데이터를 뽑아오는 기초 과정을 실습합니다.

In [1]:
# 1. 실습에 필요한 파이썬 MCP 라이브러리 (mcp) 가 설치되어 있어야 합니다.
# pip install mcp
import asyncio
from mcp.server import Server
from mcp.types import Tool, TextContent, CallToolResult
import json

## 3. 초간단 나만의 MCP 서버 만들기
이 서버는 오직 `get_weather`라는 하나의 도구(Tool)만 세상에 공개합니다.

In [2]:
# 서버 인스턴스 생성
app = Server("Weather_MCP_Server")

# 클라이언트(LLM)가 "이 서버에는 무슨 도구가 있나요?" 라고 물었을 때 대답할 목록
@app.list_tools()
async def list_tools() -> list[Tool]:
    return [
        Tool(
            name="get_weather",
            description="주어진 도시의 날씨를 알려줍니다.",
            inputSchema={
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "도시 이름 (예: Seoul, Tokyo)"}
                },
                "required": ["city"]
            }
        )
    ]

# 클라이언트(LLM)가 특정 도구를 실행해달라고 요청(Call)했을 때 작동할 로직
@app.call_tool()
async def call_tool(name: str, arguments: dict) -> list[TextContent]:
    if name == "get_weather":
        city = arguments.get("city")
        # 실무에서는 여기서 실제 기상청 API를 호출합니다.
        fake_db = {"Seoul": "맑음, 25도", "Tokyo": "비, 20도", "New York": "눈, -2도"}
        result = fake_db.get(city, "해당 도시의 날씨 정보를 찾을 수 없습니다.")
        
        return [TextContent(type="text", text=result)]
    else:
        raise ValueError(f"알 수 없는 도구입니다: {name}")

print("✅ 가상의 MCP 서버 세팅 완료!")
# 실제 실무 환경에서는 이 app을 stdio나 sse 프로토콜을 이용해 백그라운드 프로세스로 띄웁니다.
# (다음 노트북에서 실제 띄우는 방법을 다룹니다)

✅ 가상의 MCP 서버 세팅 완료!


## 4. 왜 MCP가 혁명적인가?

과거에는 우리가 LangChain에서 `get_weather`라는 파이썬 함수를 짜서 Agent에게 직접 던져주어야 했습니다.
하지만 이제는 **[기상청이 스스로 MCP 규격에 맞춘 서버를 오픈]**하기만 하면, 전 세계의 수많은 AI 에이전트(Claude, GPT, Gemini 등)가 기상청 서버 URL만 입력해서 즉시 기상청 날씨 데이터를 꺼내다 쓸 수 있게 됩니다.

즉, **"코드 베이스와 AI 에이전트 간의 완벽한 분리(Decoupling)"**가 이루어집니다.

다음 노트북 `02_mcp_custom_server.ipynb` 에서는 이 MCP 서버를 실제로 로컬 백그라운드에 띄우고(stdio 통신), 랭체인 에이전트(클라이언트)가 여기에 접속해서 파일을 읽고 쓰는 실무 연동을 해보겠습니다.